# Bias and Variance in a Banking Prediction Model

**Dataset:** `banking_operations.csv`  
**Tools:** pandas, NumPy, Matplotlib and topic-specific statistical/ML functions  

This notebook explains the concept in simple terms and connects every calculation to banking operations.

## 1. Core ideas

**High bias** means a model is too simple and underfits. **High variance** means a model is too sensitive to its training data and overfits. A useful model balances both and performs well on unseen banking transactions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

df = pd.read_csv("banking_operations.csv")
df["Transaction_Date"] = pd.to_datetime(df["Transaction_Date"])

print("Dataset shape:", df.shape)
display(df.head())

## 2. Prepare a simple prediction task

We predict `Balance_After_Transaction` using transaction amount and encoded operational categories. This is a teaching exercise, not a production banking model.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

features = ["Amount", "Account_Type", "Transaction_Type", "Channel", "Branch_City", "Status"]
target = "Balance_After_Transaction"
X = df[features]
y = df[target]

numeric_features = ["Amount"]
categorical_features = ["Account_Type", "Transaction_Type", "Channel", "Branch_City", "Status"]

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
print("Training rows:", len(X_train), "Testing rows:", len(X_test))

## 3. Compare a simple, balanced and highly flexible model

A shallow tree is deliberately simple. A regularized linear model controls complexity. An unrestricted tree can memorize a small training dataset.

In [ ]:
models = {
    "Simple tree (possible high bias)": DecisionTreeRegressor(max_depth=1, random_state=42),
    "Ridge model (controlled complexity)": Ridge(alpha=10.0),
    "Deep tree (possible high variance)": DecisionTreeRegressor(random_state=42)
}

results = []
for model_name, model in models.items():
    pipeline = Pipeline([("preprocessor", preprocessor), ("model", model)])
    pipeline.fit(X_train, y_train)
    train_predictions = pipeline.predict(X_train)
    test_predictions = pipeline.predict(X_test)
    results.append({
        "Model": model_name,
        "Training RMSE": mean_squared_error(y_train, train_predictions) ** 0.5,
        "Testing RMSE": mean_squared_error(y_test, test_predictions) ** 0.5,
        "Training R2": r2_score(y_train, train_predictions),
        "Testing R2": r2_score(y_test, test_predictions)
    })

results_df = pd.DataFrame(results).set_index("Model")
display(results_df)

## 4. Diagnose bias and variance from errors

- High training and testing error suggests high bias.
- Very low training error but much higher testing error suggests high variance.
- Similar, reasonably low errors suggest better generalization.

In [ ]:
results_df[["Training RMSE", "Testing RMSE"]].plot(
    kind="bar", figsize=(10, 5), color=["#2F80ED", "#F2994A"]
)
plt.title("Training vs Testing Error")
plt.ylabel("RMSE - lower is better")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## 5. Cross-validation: check stability across samples

A single train/test split can be lucky or unlucky. Cross-validation repeats evaluation on different folds.

In [ ]:
cv_rows = []
for model_name, model in models.items():
    pipeline = Pipeline([("preprocessor", preprocessor), ("model", model)])
    scores = cross_validate(
        pipeline, X, y, cv=5,
        scoring="neg_root_mean_squared_error",
        return_train_score=True
    )
    cv_rows.append({
        "Model": model_name,
        "Mean Training RMSE": -scores["train_score"].mean(),
        "Mean Validation RMSE": -scores["test_score"].mean(),
        "Validation RMSE Std": (-scores["test_score"]).std()
    })

cv_results = pd.DataFrame(cv_rows).set_index("Model")
display(cv_results)

## 6. Observe tree complexity directly

Increasing maximum depth generally reduces training error. Validation error may improve initially and then worsen when the model begins learning noise.

In [ ]:
depth_results = []
for depth in range(1, 11):
    model = DecisionTreeRegressor(max_depth=depth, random_state=42)
    pipeline = Pipeline([("preprocessor", preprocessor), ("model", model)])
    pipeline.fit(X_train, y_train)
    depth_results.append({
        "Depth": depth,
        "Training RMSE": mean_squared_error(y_train, pipeline.predict(X_train)) ** 0.5,
        "Testing RMSE": mean_squared_error(y_test, pipeline.predict(X_test)) ** 0.5
    })

depth_df = pd.DataFrame(depth_results).set_index("Depth")
display(depth_df)
depth_df.plot(marker="o", figsize=(8, 4))
plt.title("Model Complexity: Bias-Variance Trade-off")
plt.ylabel("RMSE")
plt.show()

## Banking interpretation and cautions

Choose complexity using validation evidence, not training accuracy alone. Production controls should also include representative data, leakage checks, drift monitoring, explainability and fairness testing. Statistical model bias is different from unfair social bias; both require separate evaluation.